# Test Qwen3 press-release embeddings

This notebook downloads `Qwen/Qwen3-Embedding-0.6B` from the Hugging Face Hub and computes normalized embeddings for the first three press releases. The model download is cached locally after the first run.

In [1]:
# Dependencies are installed in the torch-gpu conda environment.
# Do not upgrade compiled packages from inside a running notebook kernel.
import sys
print("Notebook Python:", sys.executable)
assert "envs/torch-gpu" in sys.executable, (
    "Select the torch-gpu kernel before continuing. "
    f"Current Python: {sys.executable}"
)

Notebook Python: /opt/homebrew/Caskroom/miniconda/base/envs/torch-gpu/bin/python


Dependencies should be installed from a terminal after activating `torch-gpu`. Restart the kernel after any installation or upgrade.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import sentence_transformers
import torch
import transformers
from sentence_transformers import SentenceTransformer

print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)

/opt/homebrew/Caskroom/miniconda/base/envs/torch-gpu/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/homebrew/Caskroom/miniconda/base/envs/torch-gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers: 5.6.0
transformers: 5.14.1
torch: 2.6.0.dev20241112


In [22]:
WORKING_DIRECTORY = Path(
    "/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies"
)
INPUT_PATH = WORKING_DIRECTORY / "Data/05_combine_all_states.csv"
OUTPUT_PATH = (
    WORKING_DIRECTORY
    / "Analysis/Testing/Results/qwen_embeddings_test.parquet"
)
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
PROMPT = None

data = pd.read_csv(INPUT_PATH)
test_data = data.head(3).copy()
# Keep a stable link to the row in the source CSV.
test_data.insert(0, "source_row", test_data.index)

display(test_data[["source_row", "Date", "Title", "State", "Agency"]])

,source_row,Date,Title,State,Agency
0,0,2022-06-24,10 California Communities Awarded $17 Million ...,CA,Governor
1,1,2020-03-25,12:45 PM: Governor Newsom to Make Major Announ...,CA,Governor
2,2,2021-02-08,2021 Media Credentialing Information,CA,Governor


In [39]:
data

,Title,Text,State,Agency,Date
0,10 California Communities Awarded $17 Million ...,SACRAMENTO – As part of Governor Gavin Newsom’...,CA,Governor,2022-06-24
1,12:45 PM: Governor Newsom to Make Major Announ...,SACRAMENTO – Governor Gavin Newsom will today ...,CA,Governor,2020-03-25
2,2021 Media Credentialing Information,Information regarding 2021 Governor’s Office m...,CA,Governor,2021-02-08
3,$240 Million Available to Resolve Encampments ...,SACRAMENTO – California is cleaning up encampm...,CA,Governor,2022-12-01
4,3:30 PM: Governor Newsom to Provide Update on ...,SACRAMENTO – Governor Gavin Newsom and state h...,CA,Governor,2020-03-23
...,...,...,...,...,...
15879,UVA Health Scheduling COVID Vaccine Appointmen...,Following federal government authorization of ...,VA,University,2021-05-12
15880,UVA Health to Require COVID-19 Vaccination for...,With COVID-19 cases surging across the commonw...,VA,University,2021-08-25
15881,UVA Health Joins National Trial Testing Medica...,UVA Health has joined a nationwide study evalu...,VA,University,2022-02-21
15882,"UVA Tests Drug, Given to Trump, to See If It C...",The COVID-19 study team included Bridgette Arl...,VA,University,2020-10-08


In [23]:
test_data

,source_row,Title,Text,State,Agency,Date
0,0,10 California Communities Awarded $17 Million ...,SACRAMENTO – As part of Governor Gavin Newsom’...,CA,Governor,2022-06-24
1,1,12:45 PM: Governor Newsom to Make Major Announ...,SACRAMENTO – Governor Gavin Newsom will today ...,CA,Governor,2020-03-25
2,2,2021 Media Credentialing Information,Information regarding 2021 Governor’s Office m...,CA,Governor,2021-02-08


In [24]:
# Embed the title and body together. Missing titles or bodies become empty strings.
test_data["embedding_text"] = (
    test_data["Title"].fillna("").str.strip()
    + "\n\n"
    + test_data["Text"].fillna("").str.strip()
).str.strip()

if test_data["embedding_text"].eq("").any():
    empty_rows = test_data.loc[test_data["embedding_text"].eq(""), "source_row"].tolist()
    raise ValueError(f"No title or text in source rows: {empty_rows}")

texts = test_data["embedding_text"].tolist()
for row_id, text in zip(test_data["source_row"], texts):
    print(f"Source row {row_id}: {len(text):,} characters; preview: {text[:200]!r}")

Source row 0: 1,645 characters; preview: '10 California Communities Awarded $17 Million to Address Family Homelessness\n\nSACRAMENTO – As part of Governor Gavin Newsom’s $14 billion package to address homelessness, ten California communities fr'
Source row 1: 1,088 characters; preview: '12:45 PM: Governor Newsom to Make Major Announcement to Help Families During COVID-19 Outbreak\n\nSACRAMENTO – Governor Gavin Newsom will today make a major announcement to help families during the COVI'
Source row 2: 1,179 characters; preview: '2021 Media Credentialing Information\n\nInformation regarding 2021 Governor’s Office media credentials: Working members of the media who regularly cover the Governor and wish to apply for credentials or'


In [25]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Using device:", device)
# The first run downloads the model; later runs use the Hugging Face cache.
model = SentenceTransformer(MODEL_NAME, device=device)

print("Maximum sequence length:", model.max_seq_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Model device:", model.device)

Using device: mps


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5421.84it/s]


Maximum sequence length: 32768
Embedding dimension: 1024
Model device: mps:0


/var/folders/mn/f627nbxj7x51451kv1_wnqww0000gn/T/ipykernel_49464/1021458408.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [27]:
# Check lengths with the model's tokenizer before encoding so truncation is visible.
test_data["token_count"] = [
    len(model.tokenizer.encode(text, add_special_tokens=True))
    for text in texts
]

too_long = test_data["token_count"] > model.max_seq_length
display(test_data[["source_row", "Title", "token_count"]])

if too_long.any():
    raise ValueError(
        "At least one release exceeds the model context window:\n"
        + test_data.loc[too_long, ["source_row", "token_count"]].to_string(index=False)
    )
print("All three releases fit within the model context window.")

,source_row,Title,token_count
0,0,10 California Communities Awarded $17 Million ...,311
1,1,12:45 PM: Governor Newsom to Make Major Announ...,251
2,2,2021 Media Credentialing Information,230


All three releases fit within the model context window.


In [28]:
# No query prompt is used: this is symmetric document-to-document similarity.
embeddings = model.encode(
    texts,
    batch_size=3,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
    prompt = PROMPT
)

print("Embedding matrix shape:", embeddings.shape)
print("Row norms:", np.linalg.norm(embeddings, axis=1))

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]

Embedding matrix shape: (3, 1024)
Row norms: [0.99808747 1.0009148  0.99949783]


In [42]:
embeddings.shape
test_data["source_row"]



0    0
1    1
2    2
Name: source_row, dtype: int64

In [47]:
embeddings_df = pd.DataFrame(
    embeddings,
    index=test_data["Title"],
)
embeddings_df

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
Title,,,,,,,,,,,,,,,,,,,,,
10 California Communities Awarded $17 Million to Address Family Homelessness,-0.016479,0.026367,-0.006805,0.018311,0.021118,-0.134766,-0.026489,-0.030396,0.017578,0.029785,...,0.000912,0.025024,0.012268,0.044434,0.009644,-0.086426,-0.073242,-0.017578,0.031494,-0.042725
12:45 PM: Governor Newsom to Make Major Announcement to Help Families During COVID-19 Outbreak,-0.039795,0.073242,-0.006989,-0.011292,0.092285,-0.096191,0.019287,0.000000,-0.058838,0.005463,...,-0.023682,0.018555,0.037109,0.010498,-0.012695,-0.042480,-0.000500,-0.027466,0.010620,-0.011597
2021 Media Credentialing Information,0.023926,-0.040771,-0.006226,-0.011902,0.020874,-0.071777,0.041016,0.033203,-0.041748,0.006714,...,0.032715,0.027100,0.037109,0.006714,-0.019653,0.046387,0.038086,-0.024780,0.003769,-0.048584


In [20]:
print(model.default_prompt_name)

None


In [29]:
model.prompts

{'query': 'Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:',
 'document': ''}

In [15]:
# Dot products equal cosine similarity because the rows are normalized.
similarity_matrix = embeddings @ embeddings.T
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=test_data["Title"],
    columns=test_data["Title"],
)
display(similarity_df)

Title,10 California Communities Awarded $17 Million to Address Family Homelessness,12:45 PM: Governor Newsom to Make Major Announcement to Help Families During COVID-19 Outbreak,2021 Media Credentialing Information
Title,,,
10 California Communities Awarded $17 Million to Address Family Homelessness,0.996178,0.485244,0.212856
12:45 PM: Governor Newsom to Make Major Announcement to Help Families During COVID-19 Outbreak,0.485244,1.001830,0.493978
2021 Media Credentialing Information,0.212856,0.493978,0.998995


In [16]:
embeddings

array([[-0.01647949,  0.02636719, -0.00680542, ..., -0.01757812,
         0.03149414, -0.04272461],
       [-0.03979492,  0.07324219, -0.00698853, ..., -0.02746582,
         0.01062012, -0.01159668],
       [ 0.02392578, -0.04077148, -0.00622559, ..., -0.02478027,
         0.00376892, -0.04858398]], shape=(3, 1024), dtype=float32)

In [49]:
# Save one numeric column per embedding dimension for straightforward use in R.
embedding_columns = [
    f"embedding_{i:04d}" for i in range(1, embeddings.shape[1] + 1)
]
embedding_data = pd.DataFrame(embeddings, columns=embedding_columns)
embedding_data.insert(0, "source_row", test_data["source_row"])
embedding_data.insert(1, "Title", test_data["Title"])
embedding_data

embedding_data.to_parquet(
    "embeddings.parquet",
    engine="pyarrow",
    compression="zstd",
    index=False
)
import os
os.getcwd()


'/Users/connorrust/covidpolicies/Analysis/CosSim'

In [ ]:
# Save one numeric column per embedding dimension for straightforward use in R.
embedding_columns = [
    f"embedding_{i:04d}" for i in range(1, embeddings.shape[1] + 1)
]
embedding_data = pd.DataFrame(embeddings, columns=embedding_columns)
metadata_columns = [
    "source_row", "Date", "Title", "State", "Agency", "token_count"
]
output = pd.concat(
    [test_data[metadata_columns].reset_index(drop=True), embedding_data],
    axis=1,
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
output.to_parquet(OUTPUT_PATH, index=False)
print(f"Wrote {output.shape[0]} rows and {output.shape[1]} columns to {OUTPUT_PATH}")